In [7]:
import pandas as pd
import numpy as np
import json
import requests
import logging
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



**Step 1: Loading dataset from the drive**

In [4]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/yellow_tripdata_2015-01.csv')

In [5]:
print(df.shape)
print(df.head())

(12748986, 19)
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2015-01-15 19:05:39   2015-01-15 19:23:42                1   
1         1  2015-01-10 20:33:38   2015-01-10 20:53:28                1   
2         1  2015-01-10 20:33:38   2015-01-10 20:43:41                1   
3         1  2015-01-10 20:33:39   2015-01-10 20:35:31                1   
4         1  2015-01-10 20:33:39   2015-01-10 20:52:58                1   

   trip_distance  pickup_longitude  pickup_latitude  RateCodeID  \
0           1.59        -73.993896        40.750111           1   
1           3.30        -74.001648        40.724243           1   
2           1.80        -73.963341        40.802788           1   
3           0.50        -74.009087        40.713818           1   
4           3.00        -73.971176        40.762428           1   

  store_and_fwd_flag  dropoff_longitude  dropoff_latitude  payment_type  \
0                  N         -73.974785         40.75061

In [6]:
print("Rows and Columns:", df.shape)
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))

Rows and Columns: (12748986, 19)
Total Records: 12748986
Total Columns: 19


In [7]:
print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RateCodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12748986 entries, 0 to 12748985
Data columns (total 19 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        int64  
 4   trip_distance          float64
 5   pickup_longitude       float64
 6   pickup_latitude        float64
 7   RateCodeID             int64  
 8   store_and_fwd_flag     object 
 9   dropoff_longitude      float64
 10  dropoff_latitude       float64
 11  payment_type           int64  
 12  fare_amount            float64
 13  extra                  float64
 14  mta_tax                float64
 15  tip_amount             float64
 16  tolls_amount           float64
 17  improvement_surcharge  float64
 18  total_amount           float64
dtypes: float64(12), int64(4), object(3)
memory usage: 1.8+ GB


**Step 2: Clean Column names**

In [9]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
print(df.columns.tolist())

['vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'ratecodeid', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


**Step 3: Removing duplicates**

In [10]:
print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

Duplicates before: 383
Duplicates after: 0
Dataset shape after removing duplicates: (12748603, 19)


**Step 4: Handling null values**

In [11]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nRows containing null values:",
      df.isnull().any(axis=1).sum())

Missing values in each column:
vendorid                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
pickup_longitude         0
pickup_latitude          0
ratecodeid               0
store_and_fwd_flag       0
dropoff_longitude        0
dropoff_latitude         0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    3
total_amount             0
dtype: int64

Total missing values: 3

Rows containing null values: 3


In [12]:
#No rows missing with vendorid but in question they have asked
df = df.dropna(subset=['vendorid'])

In [13]:
# Fill missing improvement_surcharge with its median
df['improvement_surcharge'] = df['improvement_surcharge'].fillna(
    df['improvement_surcharge'].median()
)

print(df.isnull().sum())
print("Total missing values:", df.isnull().sum().sum())

vendorid                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
pickup_longitude         0
pickup_latitude          0
ratecodeid               0
store_and_fwd_flag       0
dropoff_longitude        0
dropoff_latitude         0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
dtype: int64
Total missing values: 0


**Step 5: Correcting data type of number to float**

In [14]:
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
df[numeric_columns] = df[numeric_columns].astype(float)
print(df.dtypes)

vendorid                 float64
tpep_pickup_datetime      object
tpep_dropoff_datetime     object
passenger_count          float64
trip_distance            float64
pickup_longitude         float64
pickup_latitude          float64
ratecodeid               float64
store_and_fwd_flag        object
dropoff_longitude        float64
dropoff_latitude         float64
payment_type             float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
dtype: object


**Step 6: Checking for negative values**

Here i am raising exception as said from the question

In [15]:
amount_columns = [
    'fare_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'improvement_surcharge',
    'total_amount'
]

for col in amount_columns:
    if (df[col] < 0).any():
        raise ValueError(f"Negative values found in {col}")

print("All amount validations passed!")

ValueError: Negative values found in fare_amount

In [16]:
negative_fares = df[df['fare_amount'] < 0]
print("Number of negative fare records:", len(negative_fares))
print(negative_fares[['fare_amount', 'total_amount']].head(10))

Number of negative fare records: 4064
       fare_amount  total_amount
3831          -3.5         -4.80
4924          -2.5         -5.00
10046         -2.5         -4.30
16703        -52.0        -52.80
19953         -6.8         -8.10
22989        -52.0        -72.46
23879         -2.5         -3.80
26582        -52.0        -52.80
27172         -2.5         -3.30
30188         -3.5         -4.80


**Step 7: Save Data as Parquet**

In [17]:
df.to_parquet(
    "yellow_taxi_clean.parquet",
    engine="pyarrow",
    index=False
)
print("Data saved as Paraquet")

Data saved as Paraquet


In [18]:
parquet_df = pd.read_parquet("yellow_taxi_clean.parquet")

print(parquet_df.shape)
print(parquet_df.head())

(12748603, 19)
   vendorid tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0       2.0  2015-01-15 19:05:39   2015-01-15 19:23:42              1.0   
1       1.0  2015-01-10 20:33:38   2015-01-10 20:53:28              1.0   
2       1.0  2015-01-10 20:33:38   2015-01-10 20:43:41              1.0   
3       1.0  2015-01-10 20:33:39   2015-01-10 20:35:31              1.0   
4       1.0  2015-01-10 20:33:39   2015-01-10 20:52:58              1.0   

   trip_distance  pickup_longitude  pickup_latitude  ratecodeid  \
0           1.59        -73.993896        40.750111         1.0   
1           3.30        -74.001648        40.724243         1.0   
2           1.80        -73.963341        40.802788         1.0   
3           0.50        -74.009087        40.713818         1.0   
4           3.00        -73.971176        40.762428         1.0   

  store_and_fwd_flag  dropoff_longitude  dropoff_latitude  payment_type  \
0                  N         -73.974785         40.75061

**Step 8: Logging and Exception Handling**

In [19]:
import logging

# Configure logging
logging.basicConfig(
    filename="ingestion.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

try:
    logging.info("Data ingestion started")
    logging.info("CSV file loaded successfully")
    logging.info("Column names cleaned")
    logging.info("Duplicate records removed")
    logging.info("Null values handled")
    logging.info("Numeric columns converted to float")

    # Validate negative amounts
    for col in amount_columns:
        if (df[col] < 0).any():
            raise ValueError(f"Negative values found in {col}")

    logging.info("Negative amount validation passed")

    # Save as Parquet
    df.to_parquet(
        "yellow_taxi_clean.parquet",
        engine="pyarrow",
        index=False
    )

    logging.info("Data saved as Parquet")
    logging.info("Data ingestion completed successfully")

except Exception as e:
    logging.error(f"Data ingestion failed: {e}")
    print("Error:", e)

Error: Negative values found in fare_amount


In [20]:
with open("ingestion.log", "r") as file:
    print(file.read())

2026-11-05 07:07:09,298 - INFO - Data ingestion started
2026-11-05 07:07:09,298 - INFO - CSV file loaded successfully
2026-11-05 07:07:09,298 - INFO - Column names cleaned
2026-11-05 07:07:09,298 - INFO - Duplicate records removed
2026-11-05 07:07:09,298 - INFO - Null values handled
2026-11-05 07:07:09,298 - INFO - Numeric columns converted to float
2026-11-05 07:07:09,345 - ERROR - Data ingestion failed: Negative values found in fare_amount



**From the Mail sent tasks**

**Step 1: Read CSV, JSON and Parquet using Pandas**

In [23]:
json_df = pd.read_json("/content/s1")

print("JSON data:")
print(json_df.head())
print("JSON shape:", json_df.shape)

JSON data:
   id   name   age       city  salary department
0   1  Rahul  22.0  Hyderabad   45000         IT
1   2  Priya  23.0  Bangalore   55000         HR
2   3   Amit   NaN    Chennai   60000         IT
3   4  Sneha  24.0       None   50000    Finance
4   5  Kiran  22.0  Hyderabad   45000         IT
JSON shape: (5, 6)


In [22]:
parquet_df = pd.read_parquet(
    "/content/yellow_taxi_clean.parquet"
)

print("Parquet shape:", parquet_df.shape)
print(parquet_df.head())

Parquet shape: (12748603, 19)
   vendorid tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0       2.0  2015-01-15 19:05:39   2015-01-15 19:23:42              1.0   
1       1.0  2015-01-10 20:33:38   2015-01-10 20:53:28              1.0   
2       1.0  2015-01-10 20:33:38   2015-01-10 20:43:41              1.0   
3       1.0  2015-01-10 20:33:39   2015-01-10 20:35:31              1.0   
4       1.0  2015-01-10 20:33:39   2015-01-10 20:52:58              1.0   

   trip_distance  pickup_longitude  pickup_latitude  ratecodeid  \
0           1.59        -73.993896        40.750111         1.0   
1           3.30        -74.001648        40.724243         1.0   
2           1.80        -73.963341        40.802788         1.0   
3           0.50        -74.009087        40.713818         1.0   
4           3.00        -73.971176        40.762428         1.0   

  store_and_fwd_flag  dropoff_longitude  dropoff_latitude  payment_type  \
0                  N         -73.974785  

**Step 2: Clean nulls, dates and duplicates; save as Parquet**

In [24]:
print("Pickup date type:", df['tpep_pickup_datetime'].dtype)
print("Dropoff date type:", df['tpep_dropoff_datetime'].dtype)

print("\nMissing pickup dates:",
      df['tpep_pickup_datetime'].isnull().sum())

print("Missing dropoff dates:",
      df['tpep_dropoff_datetime'].isnull().sum())

Pickup date type: object
Dropoff date type: object

Missing pickup dates: 0
Missing dropoff dates: 0


In [25]:
df['tpep_pickup_datetime'] = pd.to_datetime(
    df['tpep_pickup_datetime'],
    errors='coerce'
)

df['tpep_dropoff_datetime'] = pd.to_datetime(
    df['tpep_dropoff_datetime'],
    errors='coerce'
)

print("Pickup date type:", df['tpep_pickup_datetime'].dtype)
print("Dropoff date type:", df['tpep_dropoff_datetime'].dtype)

print("Invalid pickup dates:",
      df['tpep_pickup_datetime'].isnull().sum())

print("Invalid dropoff dates:",
      df['tpep_dropoff_datetime'].isnull().sum())

Pickup date type: datetime64[ns]
Dropoff date type: datetime64[ns]
Invalid pickup dates: 0
Invalid dropoff dates: 0


In [26]:
invalid_dates = df[
    df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']
]

print("Invalid date records:", len(invalid_dates))

Invalid date records: 297


In [27]:
df = df[
    df['tpep_dropoff_datetime'] >= df['tpep_pickup_datetime']
]

print("Remaining records:", len(df))

print(
    "Invalid date records remaining:",
    (df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']).sum()
)

Remaining records: 12748306
Invalid date records remaining: 0


In [28]:
df.to_parquet(
    "/content/yellow_taxi_cleaned_final.parquet",
    engine="pyarrow",
    index=False
)
print("Updated dataset saved successfully as Parquet!")

Updated dataset saved successfully as Parquet!


In [29]:
parquet_df = pd.read_parquet(
    "/content/yellow_taxi_cleaned_final.parquet"
)
print("Parquet shape:", parquet_df.shape)

Parquet shape: (12748306, 19)


**Step 3: Pull data from a public REST API and save as JSON**

In [31]:
url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url)
response.raise_for_status()
api_data = response.json()
print("API data fetched successfully!")
print("Total records:", len(api_data))
print(api_data[:2])

API data fetched successfully!
Total records: 100
[{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}]


In [32]:
with open("/content/api_data.json", "w") as file:
    json.dump(api_data, file, indent=4)

print("API data saved successfully as JSON!")

API data saved successfully as JSON!


In [33]:
with open("/content/api_data.json", "r") as file:
    saved_data = json.load(file)

print("Total records saved:", len(saved_data))
print("First record:", saved_data[0])

Total records saved: 100
First record: {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


**Step 4: Reusable Function with Logging and Error Handling**

In [34]:
import logging
logging.basicConfig(
    filename="/content/ingestion.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)
logging.info("Data ingestion process started")
print("Logging configured successfully!")

Logging configured successfully!


In [4]:
import logging

In [8]:
def ingest_data(input_file, output_file):
    try:
        logging.info("Reading CSV file")
        df = pd.read_csv(input_file)

        logging.info("CSV file loaded successfully")

        # Clean column names
        df.columns = (
            df.columns.str.lower()
            .str.strip()
            .str.replace(" ", "_")
        )
        logging.info("Column names cleaned")

        # Remove duplicates
        df = df.drop_duplicates()
        logging.info("Duplicate records removed")

        # Handle missing values
        df = df.dropna(subset=['vendorid'])

        for col in df.columns:
            if df[col].isnull().any():
                if pd.api.types.is_numeric_dtype(df[col]):
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = df[col].fillna("Unknown")

        logging.info("Missing values handled")

        # Convert date columns
        date_columns = [
            'tpep_pickup_datetime',
            'tpep_dropoff_datetime'
        ]

        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

        logging.info("Date columns converted")

        # Save as Parquet
        df.to_parquet(output_file, engine="pyarrow", index=False)

        logging.info("Data saved as Parquet successfully")
        print("Ingestion completed successfully!")

        return df

    except Exception as e:
        logging.error(f"Data ingestion failed: {e}")
        print("Error:", e)

In [9]:
cleaned_df = ingest_data(
    "/content/drive/MyDrive/yellow_tripdata_2015-01.csv",
    "/content/drive/MyDrive/taxi_output.parquet"
)

Ingestion completed successfully!


In [10]:
print("Final shape:", cleaned_df.shape)
print(cleaned_df.head())

Final shape: (12748603, 19)
   vendorid tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2015-01-15 19:05:39   2015-01-15 19:23:42                1   
1         1  2015-01-10 20:33:38   2015-01-10 20:53:28                1   
2         1  2015-01-10 20:33:38   2015-01-10 20:43:41                1   
3         1  2015-01-10 20:33:39   2015-01-10 20:35:31                1   
4         1  2015-01-10 20:33:39   2015-01-10 20:52:58                1   

   trip_distance  pickup_longitude  pickup_latitude  ratecodeid  \
0           1.59        -73.993896        40.750111           1   
1           3.30        -74.001648        40.724243           1   
2           1.80        -73.963341        40.802788           1   
3           0.50        -74.009087        40.713818           1   
4           3.00        -73.971176        40.762428           1   

  store_and_fwd_flag  dropoff_longitude  dropoff_latitude  payment_type  \
0                  N         -73.974785    